# 05 Gold: `kpi_daily`

One row per day for the 90 days ending **2026-11-06** (the project's `as_of_date`; the synthetic data is anchored to it, not to `current_date()`), keyed by `(project_id, snapshot_date)`. It feeds the app's KPI strip (Schedule / Budget / Safety / Open NCRs), the daily-brief narrator, the Genie space and the Lakebase synced table. Pure SQL, **no AI calls**.

Every row is computed "as of that snapshot_date" from the silver tables `schedule`, `cost`, `ncrs`, `safety_log`, `daily_reports` and the gold table `compliance_findings`. Nothing from the project story is typed into the table: the story numbers (6 of 11 key milestones, MS-281 four days late, +2.3% EAC, 0 TRIR, 24 open NCRs, 238/187/51 drawings) appear only as the **asserts** further down, which check them against what the data computed.

## Definitions

| Metric | Definition | Point in time? |
|---|---|---|
| **Key milestone** | ``schedule.criticality = 5``: the 11 gate milestones (Site Preparation Complete ... Commissioning Start). Criticality 1-4 are ordinary work packages. | static set |
| `milestones_hit` | key milestones with `actual_end <= snapshot_date` | yes |
| `milestones_total` | key milestones (11) | static |
| **Open key milestone** | key milestone with no `actual_end`, or an `actual_end` after the snapshot date | yes |
| `slipped_milestone_id`, `slipped_days` | the open key milestone with the largest `snapshot_date - planned_end` where that is positive (ties: lowest id). NULL / 0 when none is late. `schedule.slip_days` is not used because it is only filled for finished milestones. | yes |
| `schedule_status` | `behind` if `slipped_days >= 10`; `at_risk` if `slipped_days` 5 to 9; else `on_track`. (A 4-day slip on one gate is "on track, with a flag", as the dashboard mock-up reads.) | yes |
| `budget_total_usd`, `budget_overrun_usd`, `budget_eac_pct` | `sum(budget_usd)`, `sum(eac_usd) - sum(budget_usd)`, and that overrun / budget x 100 (rounded to 1 decimal), from the latest `cost` report | **no** (see below) |
| **Open NCR** | `ncrs.raised_date <= snapshot_date` and (`closed_date` empty or `closed_date > snapshot_date`). `open_ncrs_compliance` / `open_ncrs_workmanship` split by `category`. | yes |
| `safety_days_without_incident` | days since the latest `safety_log` row with `is_incident = true` (first aid, recordable, lost time, property damage, environmental) on or before the snapshot date. Near misses, unsafe acts and stop-work events are not incidents. | yes |
| **Recordable** | `event_type` in (`recordable`, `lost_time`) in the 28 days ending on the snapshot date (`snapshot_date - 27` .. `snapshot_date`). `recordables_28d` is that count. | yes |
| `trir_28d` | OSHA-style: `recordables_28d x 200,000 / hours_28d`, where `hours_28d` = sum of `daily_reports.headcount` for report days in the same window x 10 hours per person-day (an assumption; the data has headcount, not hours). 0 when there are no recordables. | yes |
| `compliance_*_week` | drawing-level counts from `compliance_findings` with `review_date` in the 7 days ending on the snapshot date. A drawing **fails** if it has at least one FAIL row; otherwise it is **review** if it has a REVIEW row; otherwise it **passed**. `compliance_reviewed_week` = passed + failed + review. | yes |
| `is_current` | true only on the `as_of_date` row | |

**Not point-in-time (approximation):** the `cost` table is a single snapshot (all 120 lines carry `report_date = 2026-11-06`), so `budget_total_usd`, `budget_overrun_usd` and `budget_eac_pct` repeat the latest report's values on every day. The 90 budget values are therefore flat; the trend in the KPI strip comes from the schedule, safety, NCR and compliance columns. If a cost history is added later, replace the `cost_latest` CTE with a per-day lookup.

Everything else is computed per snapshot date from dated events. Compliance review dates only start on 2026-06-01, which is before the 90-day window, so every window is covered.

In [1]:
import contextlib
import io
import re
from pathlib import Path

from pyspark.sql import functions as F

# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("as_of_date", "2026-11-06", "As-of date (the data is anchored here, not to today)")
catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")
AS_OF = dbutils.widgets.get("as_of_date")
DAYS = 90
PROJECT_ID = "BRL-PH2"
HOURS_PER_PERSON_DAY = 10

SILVER = f"{catalog}.{prefix}_silver_engineering"
GOLD = f"{catalog}.{prefix}_gold_pm"
TARGET = f"{GOLD}.kpi_daily"
print("as_of_date:", AS_OF, "| target:", TARGET)

as_of_date: 2026-11-06 | target: mfg_mc_se_sa.bluerridge_gold_pm.kpi_daily


## Build

One query, one CTE per source. The silver columns are already typed (dates, ints, a boolean `is_incident`); an open NCR has a NULL `closed_date` and an unfinished milestone a NULL `actual_end`.

In [2]:
BUILD_SQL = f'''
WITH days AS (
  SELECT explode(sequence(date_sub(DATE'{AS_OF}', {DAYS - 1}), DATE'{AS_OF}', INTERVAL 1 DAY)) AS d
),
key_ms AS (  -- key milestones = criticality 5
  SELECT milestone_id, planned_end AS planned_end, actual_end AS actual_end
  FROM {SILVER}.schedule WHERE criticality = 5
),
ms_hit AS (
  SELECT d, count(CASE WHEN actual_end <= d THEN 1 END) AS milestones_hit, count(*) AS milestones_total
  FROM days CROSS JOIN key_ms GROUP BY d
),
ms_slip AS (  -- worst open, late key milestone as of d
  SELECT d, milestone_id AS slipped_milestone_id, datediff(d, planned_end) AS slipped_days
  FROM (
    SELECT d, milestone_id, planned_end,
           row_number() OVER (PARTITION BY d ORDER BY datediff(d, planned_end) DESC, milestone_id) AS rn
    FROM days CROSS JOIN key_ms
    WHERE (actual_end IS NULL OR actual_end > d) AND planned_end < d
  ) WHERE rn = 1
),
cost_latest AS (  -- the cost table is a single report, so this is not point-in-time (see definitions)
  SELECT sum(budget_usd) AS budget_total_usd,
         sum(eac_usd) - sum(budget_usd) AS budget_overrun_usd
  FROM {SILVER}.cost WHERE report_date = (SELECT max(report_date) FROM {SILVER}.cost)
),
ncr AS (
  SELECT d,
    count(*) AS open_ncrs,
    count(CASE WHEN category = 'compliance' THEN 1 END) AS open_ncrs_compliance,
    count(CASE WHEN category = 'workmanship' THEN 1 END) AS open_ncrs_workmanship
  FROM days JOIN (
    SELECT category, raised_date AS raised, closed_date AS closed FROM {SILVER}.ncrs
  ) n ON n.raised <= d AND (n.closed IS NULL OR n.closed > d)
  GROUP BY d
),
saf AS (
  SELECT event_date AS ev, event_type, is_incident FROM {SILVER}.safety_log
),
last_inc AS (
  SELECT d, max(ev) AS last_incident FROM days JOIN saf ON saf.is_incident AND ev <= d GROUP BY d
),
rec AS (
  SELECT d, count(*) AS recordables_28d
  FROM days JOIN saf ON event_type IN ('recordable', 'lost_time') AND ev > date_sub(d, 28) AND ev <= d GROUP BY d
),
hrs AS (
  SELECT d, sum(headcount) * {HOURS_PER_PERSON_DAY} AS hours_28d
  FROM days JOIN {SILVER}.daily_reports r ON r.report_date > date_sub(d, 28) AND r.report_date <= d GROUP BY d
),
drawing_week AS (  -- one row per (snapshot, drawing) reviewed in the 7 days ending d
  SELECT d, drawing_id, max(status = 'FAIL') AS has_fail, max(status = 'REVIEW') AS has_review
  FROM days JOIN {GOLD}.compliance_findings f ON f.review_date BETWEEN date_sub(d, 6) AND d
  GROUP BY d, drawing_id
),
comp AS (
  SELECT d,
    count(*) AS reviewed,
    count(CASE WHEN has_fail THEN 1 END) AS failed,
    count(CASE WHEN NOT has_fail AND has_review THEN 1 END) AS review,
    count(CASE WHEN NOT has_fail AND NOT has_review THEN 1 END) AS passed
  FROM drawing_week GROUP BY d
),
joined AS (
  SELECT days.d, h.milestones_hit, h.milestones_total, s.slipped_milestone_id, coalesce(s.slipped_days, 0) AS slipped_days,
         c.budget_total_usd, c.budget_overrun_usd,
         datediff(days.d, li.last_incident) AS safety_days_without_incident,
         coalesce(r.recordables_28d, 0) AS recordables_28d, hr.hours_28d,
         coalesce(n.open_ncrs, 0) AS open_ncrs, coalesce(n.open_ncrs_compliance, 0) AS open_ncrs_compliance,
         coalesce(n.open_ncrs_workmanship, 0) AS open_ncrs_workmanship,
         coalesce(cp.reviewed, 0) AS reviewed, coalesce(cp.passed, 0) AS passed, coalesce(cp.failed, 0) AS failed, coalesce(cp.review, 0) AS review
  FROM days
  LEFT JOIN ms_hit h ON h.d = days.d LEFT JOIN ms_slip s ON s.d = days.d CROSS JOIN cost_latest c
  LEFT JOIN last_inc li ON li.d = days.d LEFT JOIN rec r ON r.d = days.d LEFT JOIN hrs hr ON hr.d = days.d
  LEFT JOIN ncr n ON n.d = days.d LEFT JOIN comp cp ON cp.d = days.d
)
SELECT
  CAST('{PROJECT_ID}' AS STRING) AS project_id,
  CAST(d AS DATE) AS snapshot_date,
  d = DATE'{AS_OF}' AS is_current,
  CASE WHEN slipped_days >= 10 THEN 'behind' WHEN slipped_days >= 5 THEN 'at_risk' ELSE 'on_track' END AS schedule_status,
  CAST(milestones_hit AS INT) AS milestones_hit,
  CAST(milestones_total AS INT) AS milestones_total,
  slipped_milestone_id,
  CAST(slipped_days AS INT) AS slipped_days,
  budget_total_usd,
  budget_overrun_usd,
  round(budget_overrun_usd / budget_total_usd * 100, 1) AS budget_eac_pct,
  CAST(safety_days_without_incident AS INT) AS safety_days_without_incident,
  CAST(recordables_28d AS INT) AS recordables_28d,
  CASE WHEN recordables_28d = 0 THEN CAST(0 AS DOUBLE) ELSE round(recordables_28d * 200000.0 / hours_28d, 2) END AS trir_28d,
  CAST(open_ncrs AS INT) AS open_ncrs,
  CAST(open_ncrs_compliance AS INT) AS open_ncrs_compliance,
  CAST(open_ncrs_workmanship AS INT) AS open_ncrs_workmanship,
  CAST(reviewed AS INT) AS compliance_reviewed_week,
  CAST(passed AS INT) AS compliance_passed_week,
  CAST(failed AS INT) AS compliance_failed_week,
  CAST(review AS INT) AS compliance_review_week,
  current_timestamp() AS computed_at
FROM joined
'''
kpi = spark.sql(BUILD_SQL)
kpi.createOrReplaceTempView("kpi_build")
print("columns:", len(kpi.columns))

# The plan check: this table is pure SQL, so the physical plan must hold no AI Query step.
# explain() only plans the query; it does not run it.
plan = io.StringIO()
with contextlib.redirect_stdout(plan):
    kpi.explain(mode="formatted")
physical_plan = re.split(r"^== (?!Physical Plan)", plan.getvalue(), flags=re.M)[0]
ai_steps = re.findall(r"^\(\d+\) AI Query.*$", physical_plan, flags=re.M)
print(f"AI Query steps in the physical plan: {len(ai_steps)}")
assert len(ai_steps) == 0, "kpi_daily must not call any AI function"

columns: 22
AI Query steps in the physical plan: 0


## Write

The table is created empty with `NOT NULL` on the key columns and an informational `PRIMARY KEY (project_id, snapshot_date)` (Unity Catalog does not enforce it; Task 16's Lakebase sync needs it declared), then filled from the build. Re-running replaces the table.

In [3]:
spark.sql(f'''
CREATE OR REPLACE TABLE {TARGET} (
  project_id STRING NOT NULL COMMENT 'Project key, BRL-PH2',
  snapshot_date DATE NOT NULL COMMENT 'The day this row describes; every metric is computed as of this date',
  is_current BOOLEAN COMMENT 'True only on the as-of date row',
  schedule_status STRING COMMENT 'on_track | at_risk | behind, from slipped_days (behind >= 10, at_risk 5-9)',
  milestones_hit INT COMMENT 'Key milestones (criticality 5) with actual_end on or before snapshot_date',
  milestones_total INT COMMENT 'Key milestones in the project',
  slipped_milestone_id STRING COMMENT 'Worst-slipping open key milestone as of snapshot_date; NULL if none is late',
  slipped_days INT COMMENT 'Days that milestone is past its planned_end; 0 if none',
  budget_total_usd DOUBLE COMMENT 'Total budget, latest cost report (not point-in-time)',
  budget_overrun_usd DOUBLE COMMENT 'EAC minus budget, latest cost report (not point-in-time)',
  budget_eac_pct DOUBLE COMMENT 'EAC vs budget in percent, e.g. 2.3 = 2.3 percent over',
  safety_days_without_incident INT COMMENT 'Days since the latest is_incident event',
  recordables_28d INT COMMENT 'recordable + lost_time events in the 28 days ending snapshot_date',
  trir_28d DOUBLE COMMENT 'recordables_28d x 200000 / hours worked in the same window (headcount x 10 h); 0 if no recordables',
  open_ncrs INT COMMENT 'NCRs raised on or before snapshot_date and not closed by it',
  open_ncrs_compliance INT COMMENT 'Open NCRs, category compliance',
  open_ncrs_workmanship INT COMMENT 'Open NCRs, category workmanship',
  compliance_reviewed_week INT COMMENT 'Distinct drawings reviewed in the 7 days ending snapshot_date = passed + failed + review',
  compliance_passed_week INT COMMENT 'Reviewed drawings with only PASS rows',
  compliance_failed_week INT COMMENT 'Reviewed drawings with at least one FAIL row',
  compliance_review_week INT COMMENT 'Reviewed drawings with a REVIEW row and no FAIL row (judge answer unusable)',
  computed_at TIMESTAMP COMMENT 'When this row was computed',
  CONSTRAINT kpi_daily_pk PRIMARY KEY (project_id, snapshot_date)
) COMMENT 'Daily project KPI snapshot, 90 days ending the as-of date. Feeds the KPI strip, daily brief, Genie and the Lakebase synced table.'
''')
spark.sql(f"INSERT INTO {TARGET} SELECT * FROM kpi_build")
tbl = spark.table(TARGET)
print("rows written:", tbl.count())
spark.sql(f"DESCRIBE TABLE {TARGET}").show(40, truncate=False)

rows written: 90
+----------------------------+---------+--------------------------------------------------------------------------------------------------+
|col_name                    |data_type|comment                                                                                           |
+----------------------------+---------+--------------------------------------------------------------------------------------------------+
|project_id                  |string   |Project key, BRL-PH2                                                                              |
|snapshot_date               |date     |The day this row describes; every metric is computed as of this date                              |
|is_current                  |boolean  |True only on the as-of date row                                                                   |
|schedule_status             |string   |on_track | at_risk | behind, from slipped_days (behind >= 10, at_risk 5-9)                        |
|mi

## Asserts

First the table's own sanity checks, then the story: the as-of row must reproduce what the synthetic-data generator calibrated (`01_synthetic_data/01_generate_project_data.ipynb`). The story numbers are typed only here, as expected values.

In [4]:
tbl = spark.table(TARGET)
rows = tbl.orderBy("snapshot_date").collect()

# --- sanity ---
assert len(rows) == DAYS, f"expected {DAYS} rows, got {len(rows)}"
assert tbl.select("project_id", "snapshot_date").distinct().count() == DAYS, "primary key (project_id, snapshot_date) is not unique"
assert str(rows[0].snapshot_date) == str(spark.sql(f"SELECT date_sub(DATE'{AS_OF}', {DAYS - 1})").first()[0])
assert str(rows[-1].snapshot_date) == AS_OF
assert [str(r.snapshot_date) for r in rows if r.is_current] == [AS_OF], "is_current must be true only on the as-of date"
hits = [r.milestones_hit for r in rows]
assert all(a <= b for a, b in zip(hits, hits[1:])), "milestones_hit must not decrease over time"
count_cols = ["milestones_hit", "milestones_total", "slipped_days", "safety_days_without_incident", "recordables_28d",
              "open_ncrs", "open_ncrs_compliance", "open_ncrs_workmanship", "compliance_reviewed_week",
              "compliance_passed_week", "compliance_failed_week", "compliance_review_week"]
for r in rows:
    for c in count_cols:
        assert r[c] is not None and r[c] >= 0, f"{c} is null or negative on {r.snapshot_date}: {r[c]}"
    assert r.compliance_reviewed_week == r.compliance_passed_week + r.compliance_failed_week + r.compliance_review_week, f"compliance split does not add up on {r.snapshot_date}"
    assert r.open_ncrs >= r.open_ncrs_compliance + r.open_ncrs_workmanship, f"NCR split exceeds total on {r.snapshot_date}"
    assert r.schedule_status in ("on_track", "at_risk", "behind")
    assert r.trir_28d is not None and r.trir_28d >= 0
assert tbl.where("project_id IS NULL OR snapshot_date IS NULL").count() == 0
print("sanity asserts passed:", len(rows), "rows,", rows[0].snapshot_date, "to", rows[-1].snapshot_date)

# --- story: the as-of row ---
cur = rows[-1]
assert cur.milestones_hit == 6 and cur.milestones_total == 11, (cur.milestones_hit, cur.milestones_total)
assert cur.slipped_milestone_id == "MS-281" and cur.slipped_days == 4, (cur.slipped_milestone_id, cur.slipped_days)
assert cur.schedule_status == "on_track"
assert cur.budget_eac_pct == 2.3, cur.budget_eac_pct
assert cur.budget_overrun_usd == 4_800_000, cur.budget_overrun_usd
assert abs(cur.budget_total_usd - 208_700_000) < 100_000, cur.budget_total_usd
assert cur.trir_28d == 0 and cur.recordables_28d == 0, (cur.trir_28d, cur.recordables_28d)
assert cur.safety_days_without_incident == 28, cur.safety_days_without_incident
assert (cur.open_ncrs, cur.open_ncrs_compliance, cur.open_ncrs_workmanship) == (24, 9, 15), (cur.open_ncrs, cur.open_ncrs_compliance, cur.open_ncrs_workmanship)
assert (cur.compliance_reviewed_week, cur.compliance_passed_week, cur.compliance_failed_week, cur.compliance_review_week) == (238, 187, 51, 0), \
    (cur.compliance_reviewed_week, cur.compliance_passed_week, cur.compliance_failed_week, cur.compliance_review_week)
print("story asserts passed: 6 of 11 key milestones, MS-281 4 days late, EAC +2.3% ($4.8M over $208.7M), TRIR 0 / 28 days, 24 open NCRs (9 compliance, 15 workmanship), 238/187/51 drawings")

sanity asserts passed: 90 rows, 2026-08-09 to 2026-11-06
story asserts passed: 6 of 11 key milestones, MS-281 4 days late, EAC +2.3% ($4.8M over $208.7M), TRIR 0 / 28 days, 24 open NCRs (9 compliance, 15 workmanship), 238/187/51 drawings


## Display

The as-of row, then the last 7 days. `kpi_today.md` holds the as-of row in the words a PM reads it with.


**Note added after this notebook was run (markdown only).** `open_ncrs` can exceed compliance plus workmanship NCRs, because other NCR categories can also be open on a given day. So the split in `kpi_today.md` (for example "27 (9 / 17)") need not sum to the total.

In [5]:
display(tbl.where("is_current"))

project_id,snapshot_date,is_current,schedule_status,milestones_hit,milestones_total,slipped_milestone_id,slipped_days,budget_total_usd,budget_overrun_usd,budget_eac_pct,safety_days_without_incident,recordables_28d,trir_28d,open_ncrs,open_ncrs_compliance,open_ncrs_workmanship,compliance_reviewed_week,compliance_passed_week,compliance_failed_week,compliance_review_week,computed_at
BRL-PH2,2026-11-06,True,on_track,6,11,MS-281,4,208700000.0,4800000.0,2.3,28,0,0.0,24,9,15,238,187,51,0,2026-11-08T16:29:43.508Z


In [6]:
display(tbl.orderBy(F.col("snapshot_date").desc()).limit(7).select(
    "snapshot_date", "schedule_status", "milestones_hit", "slipped_milestone_id", "slipped_days", "budget_eac_pct",
    "safety_days_without_incident", "trir_28d", "open_ncrs", "open_ncrs_compliance", "open_ncrs_workmanship",
    "compliance_reviewed_week", "compliance_passed_week", "compliance_failed_week"))

snapshot_date,schedule_status,milestones_hit,slipped_milestone_id,slipped_days,budget_eac_pct,safety_days_without_incident,trir_28d,open_ncrs,open_ncrs_compliance,open_ncrs_workmanship,compliance_reviewed_week,compliance_passed_week,compliance_failed_week
2026-11-06,on_track,6,MS-281,4,2.3,28,0.0,24,9,15,238,187,51
2026-11-05,on_track,6,MS-281,3,2.3,27,0.0,24,9,15,214,166,48
2026-11-04,on_track,6,MS-281,2,2.3,26,0.0,27,9,17,181,137,44
2026-11-03,on_track,6,MS-281,1,2.3,25,0.0,26,9,16,152,115,37
2026-11-02,on_track,6,,0,2.3,24,0.0,26,9,16,114,88,26
2026-11-01,on_track,6,,0,2.3,23,0.0,26,9,16,88,66,22
2026-10-31,on_track,6,,0,2.3,22,0.0,26,9,16,60,45,15


In [7]:
c = tbl.where("is_current").first()
last7 = tbl.orderBy(F.col("snapshot_date").desc()).limit(7).collect()
out = Path("outputs")
out.mkdir(exist_ok=True)  # the scratch run folder has no outputs/ directory
lines = [
    f"# KPI strip as of {c.snapshot_date}",
    "",
    f"Project {c.project_id}, from `{TARGET}` (computed {c.computed_at:%Y-%m-%d %H:%M} UTC). Every figure is computed from the silver tables; the notebook asserts them against the generator's calibrated story.",
    "",
    "| KPI | Value | Detail |",
    "|---|---|---|",
    f"| Schedule | {c.schedule_status.replace('_', ' ')} | {c.milestones_hit} of {c.milestones_total} key milestones hit; {c.slipped_milestone_id} is {c.slipped_days} days late |",
    f"| Budget | EAC {c.budget_eac_pct:+.1f}% | ${c.budget_overrun_usd / 1e6:.1f}M over a ${c.budget_total_usd / 1e6:.1f}M budget |",
    f"| Safety | TRIR {c.trir_28d:g} (28 days) | {c.safety_days_without_incident} days without an incident; {c.recordables_28d} recordables in the last 28 days |",
    f"| Open NCRs | {c.open_ncrs} | {c.open_ncrs_compliance} compliance, {c.open_ncrs_workmanship} workmanship |",
    f"| Compliance this week | {c.compliance_failed_week} failed | {c.compliance_reviewed_week} drawings reviewed, {c.compliance_passed_week} passed, {c.compliance_failed_week} failed, {c.compliance_review_week} awaiting manual review |",
    "",
    "## Last 7 days",
    "",
    "| Date | Schedule | Key milestones | Slipped | EAC | Days w/o incident | TRIR 28d | Open NCRs (comp / work) | Drawings reviewed / passed / failed |",
    "|---|---|---|---|---|---|---|---|---|",
]
for r in last7:
    slip = f"{r.slipped_milestone_id} +{r.slipped_days}d" if r.slipped_milestone_id else "none"
    lines.append(f"| {r.snapshot_date} | {r.schedule_status.replace('_', ' ')} | {r.milestones_hit}/{r.milestones_total} | {slip} | {r.budget_eac_pct:+.1f}% | {r.safety_days_without_incident} | {r.trir_28d:g} | {r.open_ncrs} ({r.open_ncrs_compliance} / {r.open_ncrs_workmanship}) | {r.compliance_reviewed_week} / {r.compliance_passed_week} / {r.compliance_failed_week} |")
lines += ["", "Budget columns repeat the latest cost report on every day (the cost table is one snapshot); all other columns are computed as of each date.", ""]
(out / "kpi_today.md").write_text("\n".join(lines))
print("\n".join(lines))

# KPI strip as of 2026-11-06

Project BRL-PH2, from `mfg_mc_se_sa.bluerridge_gold_pm.kpi_daily` (computed 2026-11-08 16:29 UTC). Every figure is computed from the silver tables; the notebook asserts them against the generator's calibrated story.

| KPI | Value | Detail |
|---|---|---|
| Schedule | on track | 6 of 11 key milestones hit; MS-281 is 4 days late |
| Budget | EAC +2.3% | $4.8M over a $208.7M budget |
| Safety | TRIR 0 (28 days) | 28 days without an incident; 0 recordables in the last 28 days |
| Open NCRs | 24 | 9 compliance, 15 workmanship |
| Compliance this week | 51 failed | 238 drawings reviewed, 187 passed, 51 failed, 0 awaiting manual review |

## Last 7 days

| Date | Schedule | Key milestones | Slipped | EAC | Days w/o incident | TRIR 28d | Open NCRs (comp / work) | Drawings reviewed / passed / failed |
|---|---|---|---|---|---|---|---|---|
| 2026-11-06 | on track | 6/11 | MS-281 +4d | +2.3% | 28 | 0 | 24 (9 / 15) | 238 / 187 / 51 |
| 2026-11-05 | on track | 6/11 | 